
### **Topic: What are Runnables in LangChain?**


---

### **1. Introduction & Recap**
*   **Recap of Last Video:** Previous videos covered **Chains** (Sequential, Parallel, Conditional).
*   **Why Runnables are Important:** To truly understand how chains work internally, you must understand **Runnables**. They are the fundamental building blocks that make chains possible.
*   **Goal of This Video:** To understand what Runnables are, why they are needed, and how they solve major problems in LangChain.

---

### **2. The Backstory: The Evolution of LangChain (Why Runnables Exist)**

#### **Phase 1: The Rise of LLMs & LangChain's Initial Goal**
*   **Context (Late 2022):** ChatGPT and OpenAI APIs were released, leading to a surge in LLM-based applications.
*   **LangChain's Initial Solution:** To unify different LLM providers (OpenAI, Anthropic, Google, etc.) under a single, easy-to-use interface. This was their first big problem solved.

#### **Phase 2: More Components, More Complexity**
*   **Observation:** Building an LLM app involves more than just calling an API. It includes many steps like loading documents, splitting text, generating embeddings, storing in vector databases, retrieval, etc.
*   **LangChain's Solution:** They created helper components for all these common tasks: `DocumentLoaders`, `TextSplitters`, `Embedding Models`, `VectorStores`, `Retrievers`, `OutputParsers`, etc.
*   **Result:** LangChain became a powerful ecosystem of components. Developers could pick these "Lego blocks" and manually connect them to build apps.

#### **Phase 3: The "Eureka!" Moment – Introducing Chains**
*   **Observation:** Developers were repeatedly writing the same code to connect components (e.g., take a prompt template, format it, send it to an LLM, get a response).
*   **Solution (Chains):** LangChain created **built-in functions (Chains)** to automate these common workflows.
    *   **Example:** The `LLMChain` could take a `PromptTemplate` and an `LLM` object. Calling `.run()` on the chain would automatically format the prompt and call the LLM, reducing manual code.
    *   **Result:** This made building apps even easier. More and more chains were created for various use cases like `RetrievalQAChain`, `SimpleSequentialChain`, `APIChain`, etc.

#### **Phase 4: The Problem – Too Many Chains!**
*   **The Downside of Success:**
    1.  **Bloated Codebase:** LangChain became huge and difficult to maintain.
    2.  **Steep Learning Curve:** New developers were overwhelmed trying to learn the dozens of different chains and when to use which one.
*   **Root Cause:** The components (`LLM`, `PromptTemplate`, `Retriever`) were **not standardized**. They all had different interfaces:
    *   `LLM` used `.predict()`
    *   `PromptTemplate` used `.format()`
    *   `Retriever` used `.get_relevant_documents()`
    *   Because they were incompatible, LangChain had to write a new, custom chain function for every new way of connecting them. This was not flexible.

#### **Phase 5: The Solution – The Runnables Revolution**
*   **The Realization:** To enable any component to connect with any other component seamlessly (like true Lego blocks), they all needed to follow a **common, standardized interface**.
*   **The Fix:** LangChain rebuilt its core components to adhere to a new, unified interface. These standardized components are called **Runnables**.

---

### **3. What are Runnables? (The Core Concepts)**

Think of Runnables as the **standardized Lego blocks** of LangChain.

1.  **Unit of Work:** Each Runnable has a specific purpose. You give it an input, it processes it, and returns an output.
2.  **Common Interface:** Every Runnable follows the same interface, meaning they all share the same set of methods.
    *   **Primary Method: `.invoke()`** - Takes an input and returns an output. This is the standard way to "run" any Runnable now.
    *   Other common methods include `.batch()` (for multiple inputs) and `.stream()` (for streaming output).
3.  **Composability (Pipelines):** Because they share the same interface, Runnables can be easily connected. The output of one Runnable can automatically become the input for the next Runnable. This allows you to create complex workflows.
4.  **Chains are also Runnables:** When you connect multiple Runnables, the resulting pipeline (a chain) itself becomes a new Runnable. This means you can connect chains together to build even larger workflows, just like combining Lego blocks to make a bigger block.

---

### **4. How Runnables Work: From Scratch Code Example**

The video demonstrates the evolution by coding from scratch:

*   **Step A: Non-Standardized Components (The Old Way)**
    *   **`FakeLLM` class:** Had a `.predict()` method.
    *   **`FakePromptTemplate` class:** Had a `.format()` method.
    *   **Manual Work:** A developer had to manually call `.format()` then `.predict()` every time.
    *   **`FakeLLMChain` (The Old Chain):** A custom class was created to combine these two. It was **not flexible**. To create a new workflow (e.g., LLM -> Parser), you'd need another new custom chain class.

*   **Step B: Standardizing with Runnables (The New Way)**
    1.  **Create an Abstract `Runnable` Class:** This class defines the standard interface, with an **abstract `.invoke()` method** that all child classes must implement.
    2.  **Make Components Inherit from `Runnable`:** Both `FakeLLM` and `FakePromptTemplate` are rewritten to inherit from the `Runnable` class.
        *   They are forced to implement the `.invoke()` method. Inside `.invoke()`, they perform their original task.
        *   The old methods (`.predict()`, `.format()`) can be kept for backward compatibility but should give a warning to use `.invoke()` instead.
    3.  **Create a `RunnableConnector` (The Universal Chain):** This class also inherits from `Runnable`.
        *   Its `__init__` takes a **list of Runnables**.
        *   Its `.invoke()` method does the magic:
            1.  Takes the initial input.
            2.  Loops through the list of Runnables.
            3.  For the first Runnable, it calls `.invoke()` with the input.
            4.  It takes the output, feeds it as the input to the next Runnable's `.invoke()`, and so on.
            5.  Returns the final output.
    4.  **Result - Ultimate Flexibility:**
        *   You can now create any workflow by simply passing a list of Runnables to the `RunnableConnector`.
        *   **Example 1 (LLM Chain):** `[prompt_template, llm]`
        *   **Example 2 (LLM + Parser):** `[prompt_template, llm, output_parser]`
        *   **Example 3 (Chaining Chains):** You can even create two separate chains (which are themselves Runnables) and then connect them: `[chain_one, chain_two]` to build a more complex pipeline.

---

### **5. Key Takeaway & Connection to Real LangChain**

*   **Summary:** Runnables are the solution to the problem of component incompatibility. By enforcing a **standard `.invoke()` interface**, they make every component composable. This eliminates the need for thousands of custom chain functions and gives developers the ultimate flexibility to build any workflow.
*   **Real LangChain Code:** If you look at the LangChain source code (e.g., the `ChatOpenAI` class), you'll see it inherits from a `BaseLanguageModel`, which eventually inherits from a base `Runnable` class. That base `Runnable` class contains the **abstract `.invoke()` method**, exactly as demonstrated in the scratch example.

### **6. What's Next?**
*   The next video will explore more practical examples and actual Runnable classes within the LangChain library.



# using abstract class and making other child of it and follow it

In [1]:
from abc import ABC,abstractmethod

In [2]:
class Runnable(ABC):
    @abstractmethod
    def invoke(input_data):
        pass

In [3]:
import random
class NakliLLM(Runnable):
  def __init__(self):
    print("LLM created")

  def invoke(self,prompt):
    response_list=[
        "Kathmandu is the capital of Nepal",
        "IPL is a cricket league",
        "AI stands for Artificial Intelligence"
    ]
    return {"response":random.choice(response_list)}
    
  def predict(self,prompt):
    response_list=[
        "Kathmandu is the capital of Nepal",
        "IPL is a cricket league",
        "AI stands for Artificial Intelligence"
    ]
    return {"response":random.choice(response_list)}



In [4]:
class NalkiPromptTemplate(Runnable):
  def __init__(self,template,input_variables):
    self.template=template
    self.input_variables=input_variables
  def invoke(self,input_dict):
    return self.template.format(**input_dict)
  def format(self,input_dict):
    return self.template.format(**input_dict)

In [5]:
class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass
    def invoke(self,input_data):
        return input_data["response"]

In [6]:
class RunnableConnector(Runnable):
    def __init__(self,runnable_list):
        self.runnable_list=runnable_list
    
    def invoke(self,input_data):
        for runnable in self.runnable_list:
            input_data=runnable.invoke(input_data)
        
        return input_data

In [7]:
template=NalkiPromptTemplate(
    template="Write a {length} poem about {topic}",
    input_variables=["length","topic"]
)
llm=NakliLLM()

LLM created


In [8]:
parser=NakliStrOutputParser()

In [9]:
chain=RunnableConnector([template,llm,parser])

In [10]:
chain.invoke({"length":"long","topic":"india"})

'IPL is a cricket league'

## now merging multiple chains

In [11]:
template1=NalkiPromptTemplate(
    template="Write a joke about {topic}",
    input_variables=["topic"]
)

In [12]:
template2=NalkiPromptTemplate(
    template="Explain the following joke {response}",
    input_variables=["response"]
)

In [13]:
llm=NakliLLM()

LLM created


In [14]:
parser=NakliStrOutputParser()

In [15]:
chain1=RunnableConnector([template1,llm])

In [ ]:
# chain1.invoke({"topic":"AI"})

{'response': 'AI stands for Artificial Intelligence'}

In [17]:
chain2=RunnableConnector([template2,llm,parser])

In [ ]:
# chain2.invoke({"response":"This is a joke"})

'AI stands for Artificial Intelligence'

In [19]:
final_chain=RunnableConnector([chain1,chain2])

In [20]:
final_chain.invoke({"topic":"cricket"})

'IPL is a cricket league'